# Punto 1 — Algoritmo ID3 aplicado a una rutina familiar
### Caso de estudio: ¿La familia sale a caminar hoy?

**Objetivo:** implementar el algoritmo ID3 (Quinlan, 1986) de forma **semiautomática**, es decir,
mostrando explícitamente cada cálculo de entropía y ganancia de información en lugar de llamar
directamente a una librería de caja negra (`sklearn`, etc.). Esto permite practicar el razonamiento
del clasificador basado en teoría de la información.

**Variables de decisión (4, según lo solicitado):**

| Variable | Significado | Valores posibles |
|---|---|---|
| `Clima` | Condición del tiempo ese día | Soleado, Lluvioso, Nublado |
| `Dia` | Tipo de día | Entre_semana, Fin_de_semana |
| `Energia` | Nivel de energía/disposición de la familia | Alta, Baja |
| `Compromisos` | Si hay compromisos/obligaciones pendientes | Ninguno, Alguno |

**Variable de clase (target):** `Caminata` → Sí / No (¿la familia sale a caminar/hacer actividad física?)


In [1]:
import pandas as pd
import math
from collections import Counter

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 1. Construcción del caso de estudio (base de datos familiar)

Los 16 registros se construyeron a partir de la rutina real de decisión de una familia:
si hay compromisos, casi nunca se camina; si llueve, tampoco; si el clima acompaña y hay energía,
se sale casi siempre; los días de poca energía dependen de si es fin de semana o entre semana.
Esta lógica **no es perfectamente determinista** a propósito, para que el ejercicio de entropía
tenga sentido (si fuera 100% determinista, la entropía inicial ya sería mínima).


In [2]:
datos = [
    ("Soleado",  "Fin_de_semana", "Alta", "Ninguno", "Si"),
    ("Soleado",  "Fin_de_semana", "Alta", "Alguno",  "No"),
    ("Soleado",  "Entre_semana",  "Alta", "Ninguno", "Si"),
    ("Soleado",  "Entre_semana",  "Baja", "Ninguno", "No"),
    ("Lluvioso", "Fin_de_semana", "Alta", "Ninguno", "No"),
    ("Lluvioso", "Entre_semana",  "Alta", "Ninguno", "No"),
    ("Lluvioso", "Fin_de_semana", "Baja", "Ninguno", "No"),
    ("Nublado",  "Fin_de_semana", "Alta", "Ninguno", "Si"),
    ("Nublado",  "Entre_semana",  "Alta", "Ninguno", "Si"),
    ("Nublado",  "Fin_de_semana", "Baja", "Ninguno", "Si"),
    ("Nublado",  "Entre_semana",  "Baja", "Ninguno", "No"),
    ("Soleado",  "Fin_de_semana", "Baja", "Ninguno", "Si"),
    ("Soleado",  "Entre_semana",  "Alta", "Alguno",  "No"),
    ("Nublado",  "Fin_de_semana", "Alta", "Alguno",  "No"),
    ("Lluvioso", "Entre_semana",  "Baja", "Alguno",  "No"),
    ("Soleado",  "Fin_de_semana", "Baja", "Alguno",  "No"),
]

columnas = ["Clima", "Dia", "Energia", "Compromisos", "Caminata"]
df = pd.DataFrame(datos, columns=columnas)
df


,Clima,Dia,Energia,Compromisos,Caminata
0,Soleado,Fin_de_semana,Alta,Ninguno,Si
1,Soleado,Fin_de_semana,Alta,Alguno,No
2,Soleado,Entre_semana,Alta,Ninguno,Si
3,Soleado,Entre_semana,Baja,Ninguno,No
4,Lluvioso,Fin_de_semana,Alta,Ninguno,No
5,Lluvioso,Entre_semana,Alta,Ninguno,No
6,Lluvioso,Fin_de_semana,Baja,Ninguno,No
7,Nublado,Fin_de_semana,Alta,Ninguno,Si
8,Nublado,Entre_semana,Alta,Ninguno,Si
9,Nublado,Fin_de_semana,Baja,Ninguno,Si


## 2. Funciones base del clasificador (entropía y ganancia de información)

Se implementan **desde cero**, sin usar `DecisionTreeClassifier` de sklearn, siguiendo la
formulación clásica de Quinlan:

$$Entropía(S) = -\sum_{i} p_i \log_2 p_i$$

$$Ganancia(S, A) = Entropía(S) - \sum_{v \in valores(A)} \frac{|S_v|}{|S|} Entropía(S_v)$$


In [3]:
def entropia(serie_clase):
    '''Entropía de Shannon de una columna de clases.'''
    conteos = Counter(serie_clase)
    total = len(serie_clase)
    ent = 0.0
    for clase, n in conteos.items():
        p = n / total
        ent -= p * math.log2(p)
    return ent


def ganancia_informacion(df, atributo, clase, mostrar=True):
    '''Ganancia de información de 'atributo' respecto a 'clase', con traza semiautomática.'''
    ent_total = entropia(df[clase])
    ent_ponderada = 0.0
    detalle = []
    for valor in sorted(df[atributo].unique()):
        subconjunto = df[df[atributo] == valor]
        p = len(subconjunto) / len(df)
        e = entropia(subconjunto[clase])
        ent_ponderada += p * e
        detalle.append((valor, len(subconjunto), round(e, 4)))
    ganancia = ent_total - ent_ponderada
    if mostrar:
        print(f"  Entropia(S) = {ent_total:.4f}")
        for valor, n, e in detalle:
            print(f"    -> {atributo}={valor:<15} n={n:<3} Entropia={e:.4f}")
        print(f"  Entropia ponderada tras dividir por '{atributo}' = {ent_ponderada:.4f}")
        print(f"  Ganancia({atributo}) = {ent_total:.4f} - {ent_ponderada:.4f} = {ganancia:.4f}\n")
    return ganancia


## 3. Paso semiautomático: entropía del conjunto completo

Antes de dividir, calculamos la entropía del nodo raíz (13 registros con clase "No" / 3... veamos la
proporción real de "Sí"/"No").


In [4]:
print("Distribucion de la clase 'Caminata':")
print(df["Caminata"].value_counts(), "\n")

ent_raiz = entropia(df["Caminata"])
print(f"Entropia(S) del conjunto completo = {ent_raiz:.4f} bits")


Distribucion de la clase 'Caminata':
Caminata
No    10
Si     6
Name: count, dtype: int64 

Entropia(S) del conjunto completo = 0.9544 bits


## 4. Paso semiautomático: ganancia de información de cada atributo

Se calcula la ganancia de información **de cada una de las 4 variables por separado**, mostrando
el detalle de la entropía condicional en cada valor, tal como se haría a mano en un ejercicio de
clase, pero automatizando la aritmética.


In [5]:
atributos = ["Clima", "Dia", "Energia", "Compromisos"]

ganancias_raiz = {}
for atributo in atributos:
    print(f"--- Ganancia de informacion para '{atributo}' ---")
    ganancias_raiz[atributo] = ganancia_informacion(df, atributo, "Caminata")

mejor = max(ganancias_raiz, key=ganancias_raiz.get)
print("Resumen de ganancias en la raiz:")
for a, g in sorted(ganancias_raiz.items(), key=lambda x: -x[1]):
    print(f"  {a:<15} Ganancia = {g:.4f}")
print(f"\n>>> Atributo raiz seleccionado por ID3: '{mejor}' (mayor ganancia de informacion)")


--- Ganancia de informacion para 'Clima' ---
  Entropia(S) = 0.9544
    -> Clima=Lluvioso        n=4   Entropia=0.0000
    -> Clima=Nublado         n=5   Entropia=0.9710
    -> Clima=Soleado         n=7   Entropia=0.9852
  Entropia ponderada tras dividir por 'Clima' = 0.7345
  Ganancia(Clima) = 0.9544 - 0.7345 = 0.2200

--- Ganancia de informacion para 'Dia' ---
  Entropia(S) = 0.9544
    -> Dia=Entre_semana    n=7   Entropia=0.8631
    -> Dia=Fin_de_semana   n=9   Entropia=0.9911
  Entropia ponderada tras dividir por 'Dia' = 0.9351
  Ganancia(Dia) = 0.9544 - 0.9351 = 0.0193

--- Ganancia de informacion para 'Energia' ---
  Entropia(S) = 0.9544
    -> Energia=Alta            n=9   Entropia=0.9911
    -> Energia=Baja            n=7   Entropia=0.8631
  Entropia ponderada tras dividir por 'Energia' = 0.9351
  Ganancia(Energia) = 0.9544 - 0.9351 = 0.0193

--- Ganancia de informacion para 'Compromisos' ---
  Entropia(S) = 0.9544
    -> Compromisos=Alguno          n=5   Entropia=0.0000
    -

## 5. Algoritmo ID3 completo (recursivo)

Ahora se generaliza el paso anterior en una función recursiva que construye el árbol completo,
imprimiendo la traza de decisión en cada nodo (parámetro `traza=True`) para conservar el carácter
semiautomático del ejercicio.


In [6]:
def id3(df, atributos, clase, nivel=0, traza=True):
    indent = "  " * nivel
    clases_presentes = df[clase].unique()

    # Caso base 1: todos los ejemplos son de la misma clase -> nodo hoja puro
    if len(clases_presentes) == 1:
        if traza:
            print(f"{indent}Hoja -> '{clases_presentes[0]}'  (nodo puro, Entropia=0)")
        return clases_presentes[0]

    # Caso base 2: no quedan atributos -> se retorna la clase mayoritaria
    if not atributos:
        mayoria = df[clase].mode()[0]
        if traza:
            print(f"{indent}Hoja (sin atributos restantes) -> clase mayoritaria: '{mayoria}'")
        return mayoria

    # Paso recursivo: elegir el atributo de mayor ganancia de informacion
    if traza:
        print(f"{indent}Nodo con {len(df)} ejemplos. Evaluando atributos {atributos}:")
    ganancias = {}
    for atributo in atributos:
        ganancias[atributo] = ganancia_informacion(df, atributo, clase, mostrar=False)
        if traza:
            print(f"{indent}  Ganancia({atributo}) = {ganancias[atributo]:.4f}")

    mejor_atributo = max(ganancias, key=ganancias.get)
    if traza:
        print(f"{indent}>>> Se divide por '{mejor_atributo}'\n")

    arbol = {mejor_atributo: {}}
    atributos_restantes = [a for a in atributos if a != mejor_atributo]

    for valor in sorted(df[mejor_atributo].unique()):
        subconjunto = df[df[mejor_atributo] == valor]
        if traza:
            print(f"{indent}Rama {mejor_atributo} = {valor}  ({len(subconjunto)} ejemplos)")
        if len(subconjunto) == 0:
            arbol[mejor_atributo][valor] = df[clase].mode()[0]
        else:
            arbol[mejor_atributo][valor] = id3(
                subconjunto, atributos_restantes, clase, nivel + 1, traza
            )
    return arbol


arbol_familia = id3(df, atributos, "Caminata")


Nodo con 16 ejemplos. Evaluando atributos ['Clima', 'Dia', 'Energia', 'Compromisos']:
  Ganancia(Clima) = 0.2200
  Ganancia(Dia) = 0.0193
  Ganancia(Energia) = 0.0193
  Ganancia(Compromisos) = 0.2710
>>> Se divide por 'Compromisos'

Rama Compromisos = Alguno  (5 ejemplos)
  Hoja -> 'No'  (nodo puro, Entropia=0)
Rama Compromisos = Ninguno  (11 ejemplos)
  Nodo con 11 ejemplos. Evaluando atributos ['Clima', 'Dia', 'Energia']:
    Ganancia(Clima) = 0.4040
    Ganancia(Dia) = 0.0518
    Ganancia(Energia) = 0.0518
  >>> Se divide por 'Clima'

  Rama Clima = Lluvioso  (3 ejemplos)
    Hoja -> 'No'  (nodo puro, Entropia=0)
  Rama Clima = Nublado  (4 ejemplos)
    Nodo con 4 ejemplos. Evaluando atributos ['Dia', 'Energia']:
      Ganancia(Dia) = 0.3113
      Ganancia(Energia) = 0.3113
    >>> Se divide por 'Dia'

    Rama Dia = Entre_semana  (2 ejemplos)
      Nodo con 2 ejemplos. Evaluando atributos ['Energia']:
        Ganancia(Energia) = 1.0000
      >>> Se divide por 'Energia'

      Rama 

## 6. Árbol resultante (estructura) y reglas equivalentes

In [7]:
import pprint
print("Arbol como estructura de diccionarios:\n")
pprint.pprint(arbol_familia)


Arbol como estructura de diccionarios:

{'Compromisos': {'Alguno': 'No',
                 'Ninguno': {'Clima': {'Lluvioso': 'No',
                                       'Nublado': {'Dia': {'Entre_semana': {'Energia': {'Alta': 'Si',
                                                                                        'Baja': 'No'}},
                                                           'Fin_de_semana': 'Si'}},
                                       'Soleado': {'Dia': {'Entre_semana': {'Energia': {'Alta': 'Si',
                                                                                        'Baja': 'No'}},
                                                           'Fin_de_semana': 'Si'}}}}}}


In [8]:
def imprimir_reglas(arbol, condiciones=None):
    if condiciones is None:
        condiciones = []
    if not isinstance(arbol, dict):
        antecedente = " Y ".join(condiciones)
        print(f"SI {antecedente}\n   ENTONCES Caminata = {arbol}\n")
        return
    atributo = next(iter(arbol))
    for valor, subarbol in arbol[atributo].items():
        imprimir_reglas(subarbol, condiciones + [f"{atributo} = {valor}"])

print("Reglas de decision inducidas por ID3:\n")
imprimir_reglas(arbol_familia)


Reglas de decision inducidas por ID3:

SI Compromisos = Alguno
   ENTONCES Caminata = No

SI Compromisos = Ninguno Y Clima = Lluvioso
   ENTONCES Caminata = No

SI Compromisos = Ninguno Y Clima = Nublado Y Dia = Entre_semana Y Energia = Alta
   ENTONCES Caminata = Si

SI Compromisos = Ninguno Y Clima = Nublado Y Dia = Entre_semana Y Energia = Baja
   ENTONCES Caminata = No

SI Compromisos = Ninguno Y Clima = Nublado Y Dia = Fin_de_semana
   ENTONCES Caminata = Si

SI Compromisos = Ninguno Y Clima = Soleado Y Dia = Entre_semana Y Energia = Alta
   ENTONCES Caminata = Si

SI Compromisos = Ninguno Y Clima = Soleado Y Dia = Entre_semana Y Energia = Baja
   ENTONCES Caminata = No

SI Compromisos = Ninguno Y Clima = Soleado Y Dia = Fin_de_semana
   ENTONCES Caminata = Si



## 7. Uso del árbol para clasificar nuevos días (predicción)

In [9]:
def predecir(arbol, ejemplo):
    if not isinstance(arbol, dict):
        return arbol
    atributo = next(iter(arbol))
    valor = ejemplo.get(atributo)
    subarbol = arbol[atributo].get(valor)
    if subarbol is None:
        return "Desconocido (combinacion no vista en el entrenamiento)"
    return predecir(subarbol, ejemplo)


nuevos_dias = [
    {"Clima": "Nublado",  "Dia": "Fin_de_semana", "Energia": "Baja", "Compromisos": "Ninguno"},
    {"Clima": "Lluvioso", "Dia": "Entre_semana",  "Energia": "Alta", "Compromisos": "Ninguno"},
    {"Clima": "Soleado",  "Dia": "Fin_de_semana", "Energia": "Alta", "Compromisos": "Alguno"},
]

for dia in nuevos_dias:
    pred = predecir(arbol_familia, dia)
    print(f"{dia}  ->  Caminata = {pred}")


{'Clima': 'Nublado', 'Dia': 'Fin_de_semana', 'Energia': 'Baja', 'Compromisos': 'Ninguno'}  ->  Caminata = Si
{'Clima': 'Lluvioso', 'Dia': 'Entre_semana', 'Energia': 'Alta', 'Compromisos': 'Ninguno'}  ->  Caminata = No
{'Clima': 'Soleado', 'Dia': 'Fin_de_semana', 'Energia': 'Alta', 'Compromisos': 'Alguno'}  ->  Caminata = No


## 8. Comentario de resultados

- **Atributo raíz:** el algoritmo seleccionó **`Compromisos`** como primer nodo de decisión, ya que
  es el atributo con mayor ganancia de información: cuando `Compromisos = Alguno`, el resultado es
  siempre "No" (subconjunto puro, entropía 0), por lo que dividir por esta variable reduce más la
  incertidumbre global que dividir por `Clima`, `Dia` o `Energia`.
- **Segundo nivel:** dentro de la rama `Compromisos = Ninguno`, el atributo `Clima` aporta la mayor
  ganancia adicional, porque `Clima = Lluvioso` genera un subconjunto puro ("No" en todos los casos),
  mientras que `Soleado` y `Nublado` todavía mezclan clases y requieren seguir dividiendo por
  `Energia` y/o `Dia`.
- **Profundidad del árbol:** el árbol no es simétrico —algunas ramas terminan en un nodo hoja muy
  rápido (p. ej. `Compromisos = Alguno` o `Clima = Lluvioso`) y otras requieren evaluar las 4
  variables. Esto es exactamente el comportamiento esperado de ID3: prioriza siempre el atributo
  que **más reduce la entropía en cada paso** (estrategia voraz / *greedy*, no óptima globalmente).
- **Limitaciones del caso de estudio:** con solo 16 registros y variables categóricas, hay
  combinaciones de valores no vistas en el entrenamiento (ver la predicción "Desconocido" si ocurre);
  esto ilustra una debilidad conocida de ID3 —no generaliza fuera de las combinaciones observadas,
  a diferencia de árboles con poda o de C4.5— y motiva la necesidad de conjuntos de entrenamiento
  más grandes o de técnicas de manejo de valores faltantes/no vistos.
- El ejercicio cumple el objetivo de practicar el clasificador basado en entropía de información de
  forma semiautomática: cada ganancia de información se calculó explícitamente antes de que el
  algoritmo decidiera el atributo de partición, en vez de delegar el cálculo completo a una librería.
